# Stage 1 — Data Cleaning (Person 1)

Clean the merged dataset and remove unusable records.

**Input:** `merged_metadata.csv`

**Outputs:**
- `outputs/cleaned_dataset.csv`
- `outputs/cleaning_report.md`

In [ ]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd().resolve()
REPO_ROOT = NOTEBOOK_DIR.parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
from pipeline.config import MERGED_METADATA_CSV, STAGE1_OUTPUT_DIR
from pipeline.stage1_cleaning.cleaning_logic import find_repo_root, run_cleaning
from pipeline.utils.io import ensure_output_dir, save_csv
from pipeline.utils.reports import distribution_markdown, write_markdown_report

REPO_ROOT = find_repo_root(NOTEBOOK_DIR)
INPUT_CSV = REPO_ROOT / 'merged_metadata.csv'
OUTPUT_DIR = REPO_ROOT / 'pipeline' / 'stage1_cleaning' / 'outputs'
ensure_output_dir(OUTPUT_DIR)
print('Repository root:', REPO_ROOT)
print('Input:', INPUT_CSV)

In [ ]:
df_raw = pd.read_csv(INPUT_CSV)
print('Initial sample count:', len(df_raw))
display(df_raw.head())
display(df_raw['diagnosis'].value_counts(dropna=False))

In [ ]:
df_clean, cleaning_stats = run_cleaning(df_raw)
print('Final sample count:', cleaning_stats['final_count'])
print('Removed by reason:')
for reason, count in cleaning_stats['removed_by_reason'].items():
    print(f'  {reason}: {count}')
display(pd.Series(cleaning_stats['final_class_distribution'], name='count'))

In [ ]:
cleaned_csv = OUTPUT_DIR / 'cleaned_dataset.csv'
report_md = OUTPUT_DIR / 'cleaning_report.md'
save_csv(df_clean, cleaned_csv)

removal_lines = '\n'.join(
    f'- {reason}: **{count}**'
    for reason, count in cleaning_stats['removed_by_reason'].items()
)
sections = {
    'Overview': (
        'Stage 1 removes unusable records and keeps only NEV, MEL, BCC, and AKIEC.'
    ),
    'Initial sample count': f"**{cleaning_stats['initial_count']}**",
    'Removed samples by reason': removal_lines,
    'Final sample count': f"**{cleaning_stats['final_count']}**",
    'Class distribution after cleaning': distribution_markdown(
        cleaning_stats['final_class_distribution']
    ),
    'Metadata cleaning applied': '\n'.join([
        '- Canonicalized `diagnosis` to uppercase target labels',
        '- Standardized sex from `gender` / `sex`',
        '- Standardized localization from `region` / `localization`',
        '- Normalized boolean metadata values',
        '- Coerced numeric fields (`age`, `fitspatrick`, diameters)',
        '- Removed invalid age and Fitzpatrick values',
    ]),
    'Output files': f'- `{cleaned_csv}`',
}
write_markdown_report(report_md, sections)
print('Saved:', cleaned_csv)
print('Saved:', report_md)